# DeepWeeds — full automated Colab run
Embedded code; GPU required; 12 epochs; selection uses validation only; test once per final model.


## 0. Cai dat + duong dan

In [ ]:
from pathlib import Path
import json, os, sys
os.chdir('/content')
PAYLOAD = json.loads('{"eval.py": "#!/usr/bin/env python3\\n\\"\\"\\"eval.py - công cụ đánh giá chính thức của Lab Day 2 (DeepWeeds).\\n\\nFile này đã hoàn chỉnh: sinh viên KHÔNG sửa công thức chỉ số. Mọi con số trong\\nresults.xlsx và báo cáo phải khớp với kết quả của file này (RUBRIC.md, mục I).\\nChỉ phụ thuộc numpy và pandas.\\n\\nĐịnh nghĩa chỉ số (README.md, mục 2.2)\\n--------------------------------------\\n- top-1 accuracy : số ảnh đoán đúng / tổng số ảnh, không trọng số theo lớp.\\n- macro-F1       : trung bình cộng F1 của 9 lớp (lớp không có dự đoán đúng có F1 = 0).\\n- balanced acc.  : trung bình recall của các lớp có ảnh trong tập.\\n- precision / recall / F1 theo lớp, ma trận nhầm lẫn (hàng = nhãn thật).\\n- ECE            : 15 bin đều theo độ tin cậy (= max softmax); bin m là (b_{m-1}, b_m].\\n                   ECE = sum_m (n_m / n) * |acc_m - conf_m|\\n- mean +- std    : qua các seed, std mẫu (ddof = 1).\\n\\nĐịnh dạng file dự đoán (mỗi seed một file)\\n------------------------------------------\\n    Filename, y_true, y_pred, p0, p1, ..., p8\\np0..p8 là xác suất softmax, thứ tự theo cột `Label` của labels.csv\\n(0 = Chinee Apple ... 7 = Snake Weed, 8 = Negatives). Tên file chứa `seed<k>`,\\nví dụ predictions/F01_seed0_test.csv. Hàm `save_predictions(...)` trong file này ghi\\nđúng định dạng đó (`from eval import save_predictions`).\\n\\nCách dùng\\n---------\\n    # 1) Tính chỉ số cho một nhóm file (một cấu hình, nhiều seed)\\n    python eval.py score --pred \\"predictions/F01_seed*_test.csv\\" \\\\\\n        --test-csv labels/test_subset0.csv --labels labels/labels.csv --tag F01\\n\\n    # 2) Tự chấm phần I của RUBRIC (chung kết so với mốc)\\n    python eval.py grade \\\\\\n        --final    \\"predictions/F01_seed*_test.csv\\" \\\\\\n        --baseline \\"predictions/T00_seed*_test.csv\\" \\\\\\n        --uncal    \\"predictions/F01_uncal_seed*_test.csv\\" \\\\\\n        --final-val \\"predictions/F01_seed*_val.csv\\" \\\\\\n        --latency-p95-ms 41.5 --latency-method proper \\\\\\n        --test-csv labels/test_subset0.csv --labels labels/labels.csv\\n\\n`--uncal` (cùng cấu hình nhưng chưa temperature scaling) và `--final-val` (dự đoán\\ntrên val) là tùy chọn; thiếu thì các ý I4(a) hoặc I4(b) ghi là \\"chưa chấm được\\".\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport glob\\nimport json\\nimport math\\nimport re\\nimport sys\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n# --------------------------------------------------------------------------- #\\n# Hằng số\\n# --------------------------------------------------------------------------- #\\nNUM_CLASSES = 9\\nECE_BINS = 15\\nPROB_SUM_TOL = 1e-3  # xác suất mỗi dòng phải cộng bằng 1 (sai số cho phép)\\n\\n# Thứ tự lớp theo cột `Label` của labels.csv (lấy từ deepweeds.py của tác giả).\\nCLASS_NAMES = [\\n    \\"Chinee Apple\\", \\"Lantana\\", \\"Parkinsonia\\", \\"Parthenium\\", \\"Prickly Acacia\\",\\n    \\"Rubber Vine\\", \\"Siam Weed\\", \\"Snake Weed\\", \\"Negatives\\",\\n]\\n\\n# Hai lớp khó nhất và mốc recall (%) từ bài báo gốc (README.md, mục 2.3).\\nHARD_CLASSES = {\\"Chinee Apple\\": 88.5, \\"Snake Weed\\": 88.8}\\n\\n# ---- Ngưỡng chấm RUBRIC mục I (TẠM THỜI: giảng viên có thể chỉnh tại đây) ---- #\\nACC_TIERS = [(95.7, 7), (95.1, 6), (94.0, 5), (92.0, 3), (90.0, 1)]  # I1: (acc %, điểm)\\nI2_MIN_DELTA = 0.01          # I2: Delta macro-F1 tối thiểu để đạt 5 điểm\\nHARD_FLOORS = [(85.0, 3), (80.0, 2)]  # I3: sàn recall (%) cho cả hai lớp khó\\nI3_MIN_POINTS = 1            # I3: có báo cáo đầy đủ nhưng dưới 80%\\nI4_MAX_VAL_TEST_GAP = 0.02   # I4(b): |macro-F1 val - macro-F1 test| tối đa\\nI5_P95_BUDGET_MS = 100.0     # I5: ngân sách p95 ở batch 1\\nMIN_SEEDS = 3\\n\\n\\n# --------------------------------------------------------------------------- #\\n# Đọc và kiểm tra file dự đoán\\n# --------------------------------------------------------------------------- #\\n@dataclass\\nclass Pred:\\n    path: str\\n    seed: int | None\\n    filenames: np.ndarray\\n    y_true: np.ndarray\\n    y_pred: np.ndarray\\n    probs: np.ndarray\\n\\n\\ndef parse_seed(path: str) -> int | None:\\n    m = re.search(r\\"seed[_-]?(\\\\d+)\\", Path(path).name)\\n    return int(m.group(1)) if m else None\\n\\n\\ndef read_pred(path: str, num_classes: int = NUM_CLASSES) -> Pred:\\n    \\"\\"\\"Đọc một file dự đoán và kiểm tra định dạng. Ném ValueError nếu sai.\\"\\"\\"\\n    df = pd.read_csv(path)\\n    prob_cols = [f\\"p{i}\\" for i in range(num_classes)]\\n    missing = [c for c in [\\"Filename\\", \\"y_true\\", \\"y_pred\\", *prob_cols] if c not in df.columns]\\n    if missing:\\n        raise ValueError(f\\"{path}: thiếu cột {missing}\\")\\n    extra_p = [c for c in df.columns if re.fullmatch(r\\"p\\\\d+\\", c) and c not in prob_cols]\\n    if extra_p:\\n        raise ValueError(f\\"{path}: có cột xác suất ngoài p0..p{num_classes - 1}: {extra_p}\\")\\n    if len(df) == 0:\\n        raise ValueError(f\\"{path}: file rỗng\\")\\n    if df[\\"Filename\\"].duplicated().any():\\n        raise ValueError(f\\"{path}: có Filename bị trùng\\")\\n\\n    probs = df[prob_cols].to_numpy(dtype=np.float64)\\n    if not np.isfinite(probs).all() or (probs < 0).any() or (probs > 1 + 1e-6).any():\\n        raise ValueError(f\\"{path}: xác suất chứa NaN, âm hoặc lớn hơn 1; hãy lưu softmax, không lưu logit\\")\\n    bad_sum = np.abs(probs.sum(1) - 1.0) > PROB_SUM_TOL\\n    if bad_sum.any():\\n        raise ValueError(f\\"{path}: {int(bad_sum.sum())} dòng có tổng xác suất khác 1 \\"\\n                         f\\"(sai số > {PROB_SUM_TOL}); hãy lưu softmax, không lưu logit\\")\\n\\n    ints = {}\\n    for col in (\\"y_true\\", \\"y_pred\\"):\\n        v = pd.to_numeric(df[col], errors=\\"coerce\\").to_numpy(dtype=np.float64)\\n        if not np.isfinite(v).all() or (v != np.round(v)).any():\\n            raise ValueError(f\\"{path}: cột {col} phải là số nguyên\\")\\n        v = v.astype(np.int64)\\n        if v.min() < 0 or v.max() >= num_classes:\\n            raise ValueError(f\\"{path}: cột {col} ngoài khoảng 0..{num_classes - 1}\\")\\n        ints[col] = v\\n\\n    argmax = probs.argmax(1)\\n    mismatch = int((argmax != ints[\\"y_pred\\"]).sum())\\n    if mismatch:\\n        raise ValueError(f\\"{path}: {mismatch} dòng có y_pred khác argmax của p0..p{num_classes - 1}\\")\\n\\n    return Pred(path, parse_seed(path), df[\\"Filename\\"].to_numpy(), ints[\\"y_true\\"], ints[\\"y_pred\\"], probs)\\n\\n\\ndef save_predictions(path: str | Path, filenames, y_true, probs, num_classes: int = NUM_CLASSES) -> Path:\\n    \\"\\"\\"Ghi một file dự đoán đúng định dạng của eval.py: Filename, y_true, y_pred, p0..p{K-1}.\\n\\n    Dùng hàm này (hoặc ghi y hệt định dạng) để tạo predictions/<exp_id>_seed<k>_<split>.csv.\\n    `probs` là XÁC SUẤT softmax (mỗi dòng cộng bằng 1), không phải logit.\\n    y_pred được tính bằng argmax của probs. Kết quả luôn qua được `read_pred`.\\n    \\"\\"\\"\\n    probs = np.asarray(probs, dtype=np.float64)\\n    y_true = np.asarray(y_true, dtype=np.int64)\\n    filenames = list(filenames)\\n    if probs.ndim != 2 or probs.shape[1] != num_classes:\\n        raise ValueError(f\\"probs phải có dạng (N, {num_classes}), nhận {probs.shape}\\")\\n    if not (len(filenames) == len(y_true) == len(probs)):\\n        raise ValueError(\\"filenames, y_true, probs phải cùng độ dài\\")\\n    if np.abs(probs.sum(1) - 1.0).max() > PROB_SUM_TOL:\\n        raise ValueError(\\"probs chưa chuẩn hoá (tổng mỗi dòng khác 1); hãy áp dụng softmax trước khi lưu\\")\\n    df = pd.DataFrame({\\"Filename\\": filenames, \\"y_true\\": y_true, \\"y_pred\\": probs.argmax(1)})\\n    for i in range(num_classes):\\n        df[f\\"p{i}\\"] = probs[:, i]\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    df.to_csv(path, index=False, float_format=\\"%.8g\\")\\n    return path\\n\\n\\ndef check_against_csv(pred: Pred, csv_path: str, what: str = \\"test\\") -> None:\\n    \\"\\"\\"Đối chiếu tên file và nhãn thật với file CSV chia sẵn (Filename, Label).\\"\\"\\"\\n    ref = pd.read_csv(csv_path)\\n    if not {\\"Filename\\", \\"Label\\"} <= set(ref.columns):\\n        raise ValueError(f\\"{csv_path}: cần cột Filename và Label\\")\\n    ref_map = dict(zip(ref[\\"Filename\\"], ref[\\"Label\\"].astype(int)))\\n    names = set(pred.filenames)\\n    if names != set(ref_map):\\n        only_pred = len(names - set(ref_map))\\n        only_ref = len(set(ref_map) - names)\\n        raise ValueError(f\\"{pred.path}: không khớp {what} CSV ({csv_path}): \\"\\n                         f\\"thừa {only_pred} ảnh, thiếu {only_ref} ảnh\\")\\n    wrong = sum(int(ref_map[f]) != int(t) for f, t in zip(pred.filenames, pred.y_true))\\n    if wrong:\\n        raise ValueError(f\\"{pred.path}: {wrong} dòng có y_true khác Label trong {csv_path}\\")\\n\\n\\ndef load_names(labels_csv: str | None) -> list[str]:\\n    \\"\\"\\"Lấy tên lớp từ labels.csv (cột Label, Species) nếu có; nếu không dùng mặc định.\\"\\"\\"\\n    if not labels_csv:\\n        return list(CLASS_NAMES)\\n    df = pd.read_csv(labels_csv)\\n    if not {\\"Label\\", \\"Species\\"} <= set(df.columns):\\n        raise ValueError(f\\"{labels_csv}: cần cột Label và Species\\")\\n    m = df.drop_duplicates(\\"Label\\").sort_values(\\"Label\\")\\n    if list(m[\\"Label\\"]) != list(range(NUM_CLASSES)):\\n        raise ValueError(f\\"{labels_csv}: Label phải là 0..{NUM_CLASSES - 1}\\")\\n    return [str(s) for s in m[\\"Species\\"]]\\n\\n\\ndef _norm(name: str) -> str:\\n    return re.sub(r\\"[^a-z]\\", \\"\\", name.lower())\\n\\n\\ndef hard_class_indices(names: list[str]) -> dict[str, int]:\\n    out = {}\\n    for hard in HARD_CLASSES:\\n        for i, n in enumerate(names):\\n            if _norm(n) == _norm(hard):\\n                out[hard] = i\\n    missing = set(HARD_CLASSES) - set(out)\\n    if missing:\\n        raise ValueError(f\\"không tìm thấy lớp {sorted(missing)} trong danh sách tên lớp {names}\\")\\n    return out\\n\\n\\ndef expand(pattern: str | list[str]) -> list[str]:\\n    patterns = [pattern] if isinstance(pattern, str) else list(pattern)\\n    files: list[str] = []\\n    for p in patterns:\\n        files.extend(glob.glob(p))\\n    files = sorted(set(files))\\n    if not files:\\n        raise FileNotFoundError(f\\"không có file nào khớp: {patterns}\\")\\n    return files\\n\\n\\n# --------------------------------------------------------------------------- #\\n# Chỉ số\\n# --------------------------------------------------------------------------- #\\ndef confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, k: int = NUM_CLASSES) -> np.ndarray:\\n    cm = np.zeros((k, k), dtype=np.int64)\\n    np.add.at(cm, (y_true, y_pred), 1)\\n    return cm\\n\\n\\ndef per_class(cm: np.ndarray) -> dict[str, np.ndarray]:\\n    tp = np.diag(cm).astype(np.float64)\\n    support = cm.sum(1).astype(np.float64)\\n    predicted = cm.sum(0).astype(np.float64)\\n    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)\\n    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)\\n    denom = precision + recall\\n    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)\\n    return {\\"precision\\": precision, \\"recall\\": recall, \\"f1\\": f1, \\"support\\": support}\\n\\n\\ndef ece_score(probs: np.ndarray, y_true: np.ndarray, bins: int = ECE_BINS) -> float:\\n    \\"\\"\\"ECE với `bins` bin đều; độ tin cậy = max softmax; bin m là (b_{m-1}, b_m].\\"\\"\\"\\n    conf = probs.max(1)\\n    correct = (probs.argmax(1) == y_true).astype(np.float64)\\n    idx = np.clip(np.ceil(conf * bins).astype(np.int64) - 1, 0, bins - 1)\\n    n = len(conf)\\n    ece = 0.0\\n    for m in range(bins):\\n        mask = idx == m\\n        if mask.any():\\n            ece += mask.sum() / n * abs(correct[mask].mean() - conf[mask].mean())\\n    return float(ece)\\n\\n\\ndef compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, probs: np.ndarray,\\n                    k: int = NUM_CLASSES, bins: int = ECE_BINS) -> dict:\\n    cm = confusion_matrix(y_true, y_pred, k)\\n    pc = per_class(cm)\\n    has_support = pc[\\"support\\"] > 0\\n    nll = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, None)).mean())\\n    return {\\n        \\"n\\": int(len(y_true)),\\n        \\"top1\\": float((y_true == y_pred).mean()),\\n        \\"macro_f1\\": float(pc[\\"f1\\"].mean()),\\n        \\"balanced_acc\\": float(pc[\\"recall\\"][has_support].mean()),\\n        \\"ece\\": ece_score(probs, y_true, bins),\\n        \\"nll\\": nll,\\n        \\"precision\\": pc[\\"precision\\"], \\"recall\\": pc[\\"recall\\"], \\"f1\\": pc[\\"f1\\"],\\n        \\"support\\": pc[\\"support\\"], \\"confusion\\": cm,\\n    }\\n\\n\\nSCALARS = (\\"top1\\", \\"macro_f1\\", \\"balanced_acc\\", \\"ece\\", \\"nll\\")\\nVECTORS = (\\"precision\\", \\"recall\\", \\"f1\\")\\n\\n\\ndef mean_std(values) -> tuple[float, float]:\\n    \\"\\"\\"mean và std mẫu (ddof=1) theo trục seed; std là NaN nếu chỉ có 1 seed.\\"\\"\\"\\n    a = np.asarray(values, dtype=np.float64)\\n    mean = a.mean(axis=0)\\n    std = a.std(axis=0, ddof=1) if len(a) > 1 else np.full(a.shape[1:], np.nan)\\n    if a.ndim == 1:\\n        return float(mean), float(std)\\n    return mean, std\\n\\n\\n@dataclass\\nclass Group:\\n    preds: list[Pred]\\n    metrics: list[dict]\\n    summary: dict  # {\\"top1\\": (mean, std), ..., \\"recall\\": (vec_mean, vec_std), ...}\\n\\n    @property\\n    def seeds(self) -> list[int | None]:\\n        return [p.seed for p in self.preds]\\n\\n\\ndef load_group(pattern, ref_csv: str | None, k: int = NUM_CLASSES, ref_what: str = \\"test\\") -> Group:\\n    files = expand(pattern)\\n    preds = [read_pred(f, k) for f in files]\\n    if ref_csv:\\n        for p in preds:\\n            check_against_csv(p, ref_csv, ref_what)\\n    base_names = set(preds[0].filenames)\\n    for p in preds[1:]:\\n        if set(p.filenames) != base_names:\\n            raise ValueError(f\\"{p.path}: tập Filename khác {preds[0].path}\\")\\n    seeds = [p.seed for p in preds if p.seed is not None]\\n    if len(seeds) != len(set(seeds)):\\n        raise ValueError(f\\"seed bị trùng giữa các file: {files}\\")\\n    metrics = [compute_metrics(p.y_true, p.y_pred, p.probs, k) for p in preds]\\n    summary = {key: mean_std([m[key] for m in metrics]) for key in (*SCALARS, *VECTORS)}\\n    return Group(preds, metrics, summary)\\n\\n\\n# --------------------------------------------------------------------------- #\\n# Định dạng đầu ra\\n# --------------------------------------------------------------------------- #\\ndef fmt(mean: float, std: float, digits: int = 4) -> str:\\n    if std is None or (isinstance(std, float) and math.isnan(std)):\\n        return f\\"{mean:.{digits}f} (1 seed)\\"\\n    return f\\"{mean:.{digits}f} ± {std:.{digits}f}\\"\\n\\n\\ndef report_group(name: str, g: Group, names: list[str]) -> str:\\n    lines = [f\\"### {name} ({len(g.preds)} seed: {g.seeds}; {g.metrics[0][\'n\']} ảnh)\\", \\"\\"]\\n    if len(g.preds) < MIN_SEEDS:\\n        lines += [f\\"> CẢNH BÁO: chỉ có {len(g.preds)} seed, yêu cầu tối thiểu {MIN_SEEDS}.\\", \\"\\"]\\n    lines += [\\"| Chỉ số | mean ± std |\\", \\"|---|---|\\"]\\n    labels = {\\"top1\\": \\"top-1 accuracy\\", \\"macro_f1\\": \\"macro-F1\\", \\"balanced_acc\\": \\"balanced accuracy\\",\\n              \\"ece\\": \\"ECE (15 bin)\\", \\"nll\\": \\"NLL\\"}\\n    for key in SCALARS:\\n        lines.append(f\\"| {labels[key]} | {fmt(*g.summary[key])} |\\")\\n    lines += [\\"\\", \\"| Lớp | Precision | Recall | F1 | Số ảnh |\\", \\"|---|---|---|---|---|\\"]\\n    pm, ps = g.summary[\\"precision\\"]\\n    rm, rs = g.summary[\\"recall\\"]\\n    fm, fs = g.summary[\\"f1\\"]\\n    for i, n in enumerate(names):\\n        lines.append(f\\"| {n} | {fmt(pm[i], ps[i], 3)} | {fmt(rm[i], rs[i], 3)} | \\"\\n                     f\\"{fmt(fm[i], fs[i], 3)} | {int(g.metrics[0][\'support\'][i])} |\\")\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef save_group(out_dir: Path, tag: str, g: Group, names: list[str]) -> None:\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    rows = []\\n    for p, m in zip(g.preds, g.metrics):\\n        rows.append({\\"file\\": Path(p.path).name, \\"seed\\": p.seed,\\n                     **{k: m[k] for k in (\\"n\\", *SCALARS)}})\\n    pd.DataFrame(rows).to_csv(out_dir / f\\"{tag}_per_seed.csv\\", index=False)\\n    pc = pd.DataFrame({\\"class\\": names})\\n    for key in VECTORS:\\n        mean, std = g.summary[key]\\n        pc[f\\"{key}_mean\\"], pc[f\\"{key}_std\\"] = mean, std\\n    pc[\\"support\\"] = g.metrics[0][\\"support\\"].astype(int)\\n    pc.to_csv(out_dir / f\\"{tag}_per_class.csv\\", index=False)\\n    cm = sum(m[\\"confusion\\"] for m in g.metrics)\\n    pd.DataFrame(cm, index=[f\\"true_{n}\\" for n in names],\\n                 columns=[f\\"pred_{n}\\" for n in names]).to_csv(out_dir / f\\"{tag}_confusion_sum.csv\\")\\n    summary = {\\"seeds\\": g.seeds,\\n               **{k: {\\"mean\\": g.summary[k][0], \\"std\\": g.summary[k][1]} for k in SCALARS},\\n               **{k: {\\"mean\\": g.summary[k][0].tolist(), \\"std\\": g.summary[k][1].tolist()} for k in VECTORS},\\n               \\"classes\\": names}\\n    (out_dir / f\\"{tag}_summary.json\\").write_text(json.dumps(summary, indent=2, ensure_ascii=False))\\n\\n\\n# --------------------------------------------------------------------------- #\\n# Lệnh `score`\\n# --------------------------------------------------------------------------- #\\ndef cmd_score(args) -> int:\\n    names = load_names(args.labels)\\n    g = load_group(args.pred, args.test_csv, ref_what=\\"test\\")\\n    if not args.test_csv:\\n        print(\\"LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\\\\n\\")\\n    print(report_group(args.tag, g, names))\\n    if args.out:\\n        save_group(Path(args.out), args.tag, g, names)\\n        print(f\\"\\\\nĐã lưu kết quả vào {args.out}/{args.tag}_*\\")\\n    return 0\\n\\n\\n# --------------------------------------------------------------------------- #\\n# Lệnh `grade` (RUBRIC mục I)\\n# --------------------------------------------------------------------------- #\\ndef points_i1(acc_pct: float) -> int:\\n    for thr, pts in ACC_TIERS:\\n        if acc_pct >= thr:\\n            return pts\\n    return 0\\n\\n\\ndef points_i2(delta: float, s: float) -> int:\\n    if not math.isfinite(delta) or delta <= 0:\\n        return 0\\n    s = 0.0 if not math.isfinite(s) else s\\n    if delta > s and delta >= I2_MIN_DELTA:\\n        return 5\\n    return 4 if delta > s else 2\\n\\n\\ndef points_i3(recalls_pct: dict[str, float]) -> int:\\n    if all(recalls_pct[c] >= ref for c, ref in HARD_CLASSES.items()):\\n        return 4\\n    low = min(recalls_pct.values())\\n    for floor, pts in HARD_FLOORS:\\n        if low >= floor:\\n            return pts\\n    return I3_MIN_POINTS\\n\\n\\ndef cmd_grade(args) -> int:\\n    names = load_names(args.labels)\\n    hard_idx = hard_class_indices(names)\\n    final = load_group(args.final, args.test_csv, ref_what=\\"test\\")\\n    base = load_group(args.baseline, args.test_csv, ref_what=\\"test\\")\\n    if not args.test_csv:\\n        print(\\"LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\\\\n\\")\\n    warnings: list[str] = []\\n    for label, g in ((\\"chung kết\\", final), (\\"mốc\\", base)):\\n        if len(g.preds) < MIN_SEEDS:\\n            warnings.append(f\\"nhóm {label} chỉ có {len(g.preds)} seed (< {MIN_SEEDS})\\")\\n    if set(final.preds[0].filenames) != set(base.preds[0].filenames):\\n        warnings.append(\\"tập Filename của chung kết và mốc khác nhau\\")\\n\\n    items: list[tuple[str, str, int | None, int, str]] = []  # (mã, tiêu chí, điểm, tối đa, ghi chú)\\n\\n    # I1\\n    acc = final.summary[\\"top1\\"][0] * 100\\n    items.append((\\"I1\\", \\"Top-1 accuracy test\\", points_i1(acc), 7, f\\"{acc:.2f}% (mean {len(final.preds)} seed)\\"))\\n\\n    # I2\\n    mf_f, sd_f = final.summary[\\"macro_f1\\"]\\n    mf_b, sd_b = base.summary[\\"macro_f1\\"]\\n    delta = mf_f - mf_b\\n    s = max([v for v in (sd_f, sd_b) if math.isfinite(v)], default=math.nan)\\n    items.append((\\"I2\\", \\"Macro-F1 cải thiện so với mốc\\", points_i2(delta, s), 5,\\n                  f\\"final {mf_f:.4f}, mốc {mf_b:.4f}, Δ={delta:+.4f}, s={s:.4f}\\"))\\n\\n    # I3\\n    rec_mean = final.summary[\\"recall\\"][0] * 100\\n    recalls = {c: float(rec_mean[i]) for c, i in hard_idx.items()}\\n    note = \\", \\".join(f\\"{c} {recalls[c]:.1f}% (mốc {HARD_CLASSES[c]}%)\\" for c in HARD_CLASSES)\\n    items.append((\\"I3\\", \\"Recall hai lớp khó\\", points_i3(recalls), 4, note))\\n\\n    # I4(a)\\n    if args.uncal:\\n        uncal = load_group(args.uncal, args.test_csv, ref_what=\\"test\\")\\n        e_after, e_before = final.summary[\\"ece\\"][0], uncal.summary[\\"ece\\"][0]\\n        items.append((\\"I4a\\", \\"ECE sau TS < ECE trước\\", int(e_after < e_before), 1,\\n                      f\\"trước {e_before:.4f}, sau {e_after:.4f}\\"))\\n    else:\\n        items.append((\\"I4a\\", \\"ECE sau TS < ECE trước\\", None, 1, \\"chưa chấm được (thiếu --uncal)\\"))\\n\\n    # I4(b)\\n    if args.final_val:\\n        fv = load_group(args.final_val, args.val_csv, ref_what=\\"val\\")\\n        gap = abs(fv.summary[\\"macro_f1\\"][0] - mf_f)\\n        items.append((\\"I4b\\", \\"Chênh macro-F1 val/test <= %.2f\\" % I4_MAX_VAL_TEST_GAP,\\n                      int(gap <= I4_MAX_VAL_TEST_GAP), 1,\\n                      f\\"val {fv.summary[\'macro_f1\'][0]:.4f}, test {mf_f:.4f}, chênh {gap:.4f}\\"))\\n    else:\\n        items.append((\\"I4b\\", \\"Chênh macro-F1 val/test <= %.2f\\" % I4_MAX_VAL_TEST_GAP, None, 1,\\n                      \\"chưa chấm được (thiếu --final-val)\\"))\\n\\n    # I5\\n    if args.latency_p95_ms is None:\\n        items.append((\\"I5\\", \\"Cấu hình thời gian thực\\", None, 2, \\"chưa chấm được (thiếu --latency-p95-ms)\\"))\\n    else:\\n        ok = args.latency_p95_ms <= I5_P95_BUDGET_MS\\n        pts = 0 if not ok else (2 if args.latency_method == \\"proper\\" else 1)\\n        items.append((\\"I5\\", \\"Cấu hình thời gian thực\\", pts, 2,\\n                      f\\"p95 = {args.latency_p95_ms:.1f} ms (ngân sách {I5_P95_BUDGET_MS:.0f} ms), \\"\\n                      f\\"đo {\'đúng cách\' if args.latency_method == \'proper\' else \'chưa đủ warmup/synchronize\'}\\"))\\n\\n    scored = [i for i in items if i[2] is not None]\\n    got = sum(i[2] for i in scored)\\n    max_scored = sum(i[3] for i in scored)\\n    lines = [\\"## Tự chấm RUBRIC mục I (đề xuất; giảng viên xác nhận)\\", \\"\\",\\n             \\"| Mã | Tiêu chí | Điểm | Tối đa | Chi tiết |\\", \\"|---|---|---|---|---|\\"]\\n    for code, crit, pts, mx, note in items:\\n        lines.append(f\\"| {code} | {crit} | {\'-\' if pts is None else pts} | {mx} | {note} |\\")\\n    lines += [\\"\\", f\\"**Tổng các ý đã chấm: {got} / {max_scored}** (phần I tối đa 20).\\"]\\n    pending = [i[0] for i in items if i[2] is None]\\n    if pending:\\n        lines.append(f\\"Chưa chấm được: {\', \'.join(pending)}.\\")\\n    if warnings:\\n        lines += [\\"\\", \\"Cảnh báo:\\"] + [f\\"- {w}\\" for w in warnings]\\n    lines += [\\"\\", \\"Ngưỡng điểm là TẠM THỜI (xem khối hằng số đầu file eval.py và RUBRIC.md mục I).\\"]\\n    print(\\"\\\\n\\".join(lines))\\n\\n    if args.out:\\n        out = Path(args.out)\\n        out.mkdir(parents=True, exist_ok=True)\\n        (out / \\"grade_I.json\\").write_text(json.dumps({\\n            \\"items\\": [{\\"code\\": c, \\"criterion\\": cr, \\"points\\": p, \\"max\\": m, \\"note\\": n} for c, cr, p, m, n in items],\\n            \\"total\\": got, \\"max_scored\\": max_scored, \\"warnings\\": warnings}, indent=2, ensure_ascii=False))\\n    return 0\\n\\n\\n# --------------------------------------------------------------------------- #\\n# CLI\\n# --------------------------------------------------------------------------- #\\ndef build_parser() -> argparse.ArgumentParser:\\n    ap = argparse.ArgumentParser(description=\\"Đánh giá Lab Day 2 (DeepWeeds)\\")\\n    sub = ap.add_subparsers(dest=\\"cmd\\", required=True)\\n\\n    sc = sub.add_parser(\\"score\\", help=\\"tính chỉ số cho một nhóm file dự đoán (nhiều seed)\\")\\n    sc.add_argument(\\"--pred\\", nargs=\\"+\\", required=True, help=\\"glob của file dự đoán\\")\\n    sc.add_argument(\\"--test-csv\\", help=\\"labels/test_subset0.csv để đối chiếu tên file và nhãn\\")\\n    sc.add_argument(\\"--labels\\", help=\\"labels/labels.csv để lấy tên lớp\\")\\n    sc.add_argument(\\"--tag\\", default=\\"score\\")\\n    sc.add_argument(\\"--out\\", help=\\"thư mục lưu JSON/CSV\\")\\n    sc.set_defaults(func=cmd_score)\\n\\n    gr = sub.add_parser(\\"grade\\", help=\\"tự chấm phần I của RUBRIC\\")\\n    gr.add_argument(\\"--final\\", nargs=\\"+\\", required=True, help=\\"glob dự đoán test của cấu hình chung kết\\")\\n    gr.add_argument(\\"--baseline\\", nargs=\\"+\\", required=True, help=\\"glob dự đoán test của mốc (T00 + I00)\\")\\n    gr.add_argument(\\"--uncal\\", nargs=\\"+\\", help=\\"glob dự đoán test của chung kết khi chưa temperature scaling\\")\\n    gr.add_argument(\\"--final-val\\", nargs=\\"+\\", help=\\"glob dự đoán val của chung kết\\")\\n    gr.add_argument(\\"--latency-p95-ms\\", type=float, help=\\"p95 batch 1 của cấu hình thời gian thực (ms)\\")\\n    gr.add_argument(\\"--latency-method\\", choices=[\\"proper\\", \\"partial\\"], default=\\"proper\\",\\n                    help=\\"proper = có warmup và synchronize; partial = chưa đủ\\")\\n    gr.add_argument(\\"--test-csv\\", help=\\"labels/test_subset0.csv\\")\\n    gr.add_argument(\\"--val-csv\\", help=\\"labels/val_subset0.csv (đối chiếu --final-val)\\")\\n    gr.add_argument(\\"--labels\\", help=\\"labels/labels.csv\\")\\n    gr.add_argument(\\"--out\\", help=\\"thư mục lưu grade_I.json\\")\\n    gr.set_defaults(func=cmd_grade)\\n    return ap\\n\\n\\ndef main(argv=None) -> int:\\n    args = build_parser().parse_args(argv)\\n    try:\\n        return args.func(args)\\n    except (ValueError, FileNotFoundError) as e:\\n        print(f\\"LỖI: {e}\\", file=sys.stderr)\\n        return 2\\n\\n\\nif __name__ == \\"__main__\\":\\n    sys.exit(main())\\n", "code/benchmark.py": "\\"\\"\\"benchmark.py - đo độ trễ đúng cách: warmup, synchronize, p50/p95/p99.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport time\\n\\nimport torch\\nimport numpy as np  # noqa: E402  (torch truoc numpy: tranh xung dot OpenMP/MKL tren Windows)\\n\\n\\ndef bench(fn, warmup: int = 10, iters: int = 100, sync=None) -> dict:\\n    \\"\\"\\"Đo mili-giây của fn(). sync()=torch.cuda.synchronize hoặc None trên CPU.\\"\\"\\"\\n    for _ in range(warmup):\\n        fn()\\n        if sync is not None:\\n            sync()\\n    ts = []\\n    for _ in range(iters):\\n        if sync is not None:\\n            sync()\\n        t0 = time.perf_counter()\\n        fn()\\n        if sync is not None:\\n            sync()\\n        ts.append((time.perf_counter() - t0) * 1000.0)\\n    a = np.asarray(ts, dtype=np.float64)\\n    return {\\"p50\\": float(np.percentile(a, 50)), \\"p95\\": float(np.percentile(a, 95)),\\n            \\"p99\\": float(np.percentile(a, 99)), \\"mean\\": float(a.mean()), \\"n\\": int(iters)}\\n\\n\\ndef latency_report(model, batch_size: int, img_size: int, dtype: str = \\"fp32\\", device: str = \\"cuda\\",\\n                   warmup: int = 10, iters: int = 100) -> dict:\\n    \\"\\"\\"Đo forward với đầu vào ngẫu nhiên. dtype: fp32 | amp | fp16.\\"\\"\\"\\n    assert dtype in (\\"fp32\\", \\"amp\\", \\"fp16\\")\\n    dev = torch.device(device if (device == \\"cuda\\" and torch.cuda.is_available()) else \\"cpu\\")\\n    model = model.eval().to(dev)\\n    use_half = dtype == \\"fp16\\" and dev.type == \\"cuda\\"\\n    if use_half:\\n        model = model.half()\\n    x = torch.randn(batch_size, 3, img_size, img_size, device=dev)\\n    if use_half:\\n        x = x.half()\\n    sync = torch.cuda.synchronize if dev.type == \\"cuda\\" else None\\n\\n    @torch.inference_mode()\\n    def fn():\\n        if dtype == \\"amp\\" and dev.type == \\"cuda\\":\\n            with torch.autocast(\\"cuda\\"):\\n                model(x)\\n        else:\\n            model(x)\\n    r = bench(fn, warmup, iters, sync)\\n    gpu = torch.cuda.get_device_name(dev) if dev.type == \\"cuda\\" else \\"CPU\\"\\n    return {\\"gpu\\": gpu, \\"dtype\\": dtype, \\"batch\\": batch_size, \\"img_size\\": img_size,\\n            \\"p50\\": r[\\"p50\\"], \\"p95\\": r[\\"p95\\"], \\"p99\\": r[\\"p99\\"],\\n            \\"images_per_s\\": batch_size / (r[\\"p50\\"] / 1000.0), \\"torch\\": torch.__version__}\\n\\n\\ndef tta_latency(model, k_views: int, **kw) -> dict:\\n    \\"\\"\\"Độ trễ TTA K view: đo thật 1 view rồi so với K*p50 (chi phí ~tuyến tính theo K).\\"\\"\\"\\n    one = latency_report(model, **kw)\\n    return {**one, \\"k_views\\": k_views, \\"p50_k\\": one[\\"p50\\"] * k_views,\\n            \\"p95_k\\": one[\\"p95\\"] * k_views, \\"note\\": \\"ước lượng K*p50; đo thật khi chạy TTA đầy đủ\\"}\\n", "code/dataset.py": "\\"\\"\\"dataset.py - đọc DeepWeeds, kiểm tra chia dữ liệu, transform, DataLoader.\\n\\nChạy trên Colab/Kaggle (có torch + torchvision). Giữ đúng giao diện starter/:\\n    load_split / check_split / build_transforms / DeepWeedsDataset / make_loader\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport random\\nfrom pathlib import Path\\n\\nimport torch\\nimport numpy as np  # noqa: E402  (torch truoc numpy: tranh xung dot OpenMP/MKL tren Windows)\\nimport pandas as pd  # noqa: E402\\nfrom PIL import Image\\n\\nNUM_CLASSES = 9\\nCLASS_NAMES = [\\n    \\"Chinee Apple\\", \\"Lantana\\", \\"Parkinsonia\\", \\"Parthenium\\", \\"Prickly Acacia\\",\\n    \\"Rubber Vine\\", \\"Siam Weed\\", \\"Snake Weed\\", \\"Negatives\\",\\n]\\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\\nIMAGENET_STD = (0.229, 0.224, 0.225)\\nTOTAL_IMAGES = 17509\\n\\n\\ndef load_split(labels_dir: str | Path, fold: int = 0):\\n    \\"\\"\\"Đọc train/val/test_subset{fold}.csv, không sửa/lọc/chia lại (S1).\\"\\"\\"\\n    labels_dir = Path(labels_dir)\\n    train_df = pd.read_csv(labels_dir / f\\"train_subset{fold}.csv\\")\\n    val_df = pd.read_csv(labels_dir / f\\"val_subset{fold}.csv\\")\\n    test_df = pd.read_csv(labels_dir / f\\"test_subset{fold}.csv\\")\\n    for name, df in ((\\"train\\", train_df), (\\"val\\", val_df), (\\"test\\", test_df)):\\n        if not {\\"Filename\\", \\"Label\\"} <= set(df.columns):\\n            raise ValueError(f\\"{name}: cần cột Filename và Label, có {list(df.columns)}\\")\\n    return train_df, val_df, test_df\\n\\n\\ndef check_split(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame,\\n                images_dir: str | Path) -> dict:\\n    \\"\\"\\"Kiểm tra bắt buộc README 2.1: số ảnh, per-class, giao rỗng, hợp 17509, file tồn tại.\\"\\"\\"\\n    images_dir = Path(images_dir)\\n    n = {\\"train\\": len(train_df), \\"val\\": len(val_df), \\"test\\": len(test_df)}\\n    total = sum(n.values())\\n    per_class = {}\\n    for name, df in ((\\"train\\", train_df), (\\"val\\", val_df), (\\"test\\", test_df)):\\n        vc = df[\\"Label\\"].value_counts().sort_index()\\n        per_class[name] = {int(k): int(v) for k, v in vc.items()}\\n    s_tr, s_va, s_te = set(train_df[\\"Filename\\"]), set(val_df[\\"Filename\\"]), set(test_df[\\"Filename\\"])\\n    overlap = {\\n        \\"train_val\\": len(s_tr & s_va),\\n        \\"train_test\\": len(s_tr & s_te),\\n        \\"val_test\\": len(s_va & s_te),\\n    }\\n    assert overlap[\\"train_val\\"] == 0, f\\"train∩val = {overlap[\'train_val\']} (phải rỗng)\\"\\n    assert overlap[\\"train_test\\"] == 0, f\\"train∩test = {overlap[\'train_test\']} (phải rỗng)\\"\\n    assert overlap[\\"val_test\\"] == 0, f\\"val∩test = {overlap[\'val_test\']} (phải rỗng)\\"\\n    assert total == TOTAL_IMAGES, f\\"hợp 3 tập = {total}, kỳ vọng {TOTAL_IMAGES}\\"\\n    for name, exp in ((\\"train\\", 0.6), (\\"val\\", 0.2), (\\"test\\", 0.2)):\\n        ratio = n[name] / total\\n        assert abs(ratio - exp) <= 0.01 + 1e-9, f\\"tỉ lệ {name}={ratio:.4f}, lệch >1pp so với {exp}\\"\\n    missing = [f for f in (s_tr | s_va | s_te) if not (images_dir / f).exists()]\\n    assert not missing, f\\"thiếu {len(missing)} file ảnh trong {images_dir}, vd: {missing[:5]}\\"\\n    print(f\\"split OK: train={n[\'train\']} val={n[\'val\']} test={n[\'test\']} total={total}\\")\\n    print(f\\"overlap: {overlap} | union={len(s_tr | s_va | s_te)}\\")\\n    return {\\"n\\": n, \\"per_class\\": per_class, \\"overlap\\": overlap, \\"total\\": total}\\n\\n\\ndef build_transforms(train: bool, img_size: int = 224, aug: str = \\"basic\\"):\\n    \\"\\"\\"Train: RandomResizedCrop + hflip (+color nếu aug>=color); Val/test: Resize256+CenterCrop, không aug ngẫu nhiên.\\"\\"\\"\\n    from torchvision import transforms\\n    normalize = transforms.Normalize(mean=list(IMAGENET_MEAN), std=list(IMAGENET_STD))\\n    if not train:\\n        return transforms.Compose([\\n            transforms.Resize(256),\\n            transforms.CenterCrop(img_size),\\n            transforms.ToTensor(),\\n            normalize,\\n        ])\\n    tf = [transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),\\n          transforms.RandomHorizontalFlip(p=0.5)]\\n    # Lật dọc KHÔNG dùng mặc định: ảnh cỏ dại có hướng trọng lực (đất/trời);\\n    # lật dọc tạo mẫu phi thực tế -> chỉ thử như ablation có kiểm soát.\\n    if aug in (\\"color\\", \\"trivial\\", \\"randaug\\"):\\n        tf.append(transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02))\\n    if aug == \\"trivial\\":\\n        tf.append(transforms.TrivialAugmentWide())\\n    elif aug == \\"randaug\\":\\n        tf.append(transforms.RandAugment(num_ops=2, magnitude=9))\\n    elif aug not in (\\"basic\\", \\"color\\", \\"trivial\\", \\"randaug\\"):\\n        raise ValueError(f\\"aug không hỗ trợ: {aug}\\")\\n    tf += [transforms.ToTensor(), normalize]\\n    return transforms.Compose(tf)\\n\\n\\nclass DeepWeedsDataset(torch.utils.data.Dataset):\\n    \\"\\"\\"Dataset đọc ảnh theo DataFrame. __getitem__ -> (tensor, int label, filename).\\"\\"\\"\\n\\n    def __init__(self, df: pd.DataFrame, images_dir: str | Path, transform=None):\\n        self.df = df.reset_index(drop=True)\\n        self.images_dir = Path(images_dir)\\n        self.transform = transform\\n\\n    def __len__(self) -> int:\\n        return len(self.df)\\n\\n    def __getitem__(self, i: int):\\n        row = self.df.iloc[i]\\n        path = self.images_dir / str(row[\\"Filename\\"])\\n        with Image.open(path) as im:\\n            img = im.convert(\\"RGB\\")\\n        if self.transform is not None:\\n            img = self.transform(img)\\n        return img, int(row[\\"Label\\"]), str(row[\\"Filename\\"])\\n\\n\\ndef seed_worker(worker_id: int):\\n    w = torch.initial_seed() % 2**32\\n    np.random.seed(w + worker_id)\\n    random.seed(w + worker_id)\\n\\n\\ndef make_loader(df: pd.DataFrame, images_dir: str | Path, transform, batch_size: int,\\n                train: bool, sampler: str | None = None, num_workers: int = 2):\\n    \\"\\"\\"train=True: shuffle (hoặc WeightedRandomSampler nếu sampler=\'balanced\'). Val/test: giữ thứ tự.\\"\\"\\"\\n    ds = DeepWeedsDataset(df, images_dir, transform)\\n    use_sampler = None\\n    shuffle = train\\n    if sampler == \\"balanced\\":\\n        counts = df[\\"Label\\"].value_counts().sort_index()\\n        w = 1.0 / counts.reindex(range(NUM_CLASSES), fill_value=1).to_numpy(dtype=np.float64)\\n        weights = torch.tensor([w[int(l)] for l in df[\\"Label\\"].to_numpy()], dtype=torch.double)\\n        use_sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights), replacement=True)\\n        shuffle = False\\n    elif sampler is not None:\\n        raise ValueError(f\\"sampler không hỗ trợ: {sampler}\\")\\n    g = torch.Generator()\\n    g.manual_seed(0)\\n    return torch.utils.data.DataLoader(\\n        ds, batch_size=batch_size, shuffle=shuffle, sampler=use_sampler,\\n        num_workers=num_workers, pin_memory=True, drop_last=(train and len(df) > batch_size),\\n        worker_init_fn=seed_worker, generator=g if num_workers == 0 else None,\\n    )\\n", "code/inference.py": "\\"\\"\\"inference.py - suy luận Bước 3: TTA, gộp view, ensemble, temperature scaling, gộp BN.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nimport numpy as np  # noqa: E402  (torch truoc numpy: tranh xung dot OpenMP/MKL tren Windows)\\n\\n\\n@torch.inference_mode()\\ndef predict_logits(model, loader, device, view=None):\\n    model.eval()\\n    names, ys, outs = [], [], []\\n    for x, y, fn in loader:\\n        if view is not None:\\n            x = view(x)\\n        x = x.to(device, non_blocking=True)\\n        outs.append(model(x).detach().cpu())\\n        ys.append(torch.as_tensor(y))\\n        names.extend(list(fn))\\n    return names, torch.cat(ys).numpy().astype(np.int64), torch.cat(outs).numpy()\\n\\n\\ndef view_identity(x):\\n    return x\\n\\n\\ndef view_hflip(x):\\n    \\"\\"\\"Lật ngang batch (N,C,H,W).\\"\\"\\"\\n    return torch.flip(x, dims=[-1])\\n\\n\\ndef views_multicrop(x, crop: int):\\n    \\"\\"\\"5 crop (4 góc + giữa) kích thước crop x crop từ ảnh vuông.\\"\\"\\"\\n    _, _, h, w = x.shape\\n    assert crop <= min(h, w), f\\"crop {crop} > ảnh {h}x{w}\\"\\n    ys = (0, h - crop, (h - crop) // 2)\\n    xs = (0, w - crop, (w - crop) // 2)\\n    views = [x[:, :, y:y + crop, x0:x0 + crop] for y in (ys[0], ys[1], ys[2])\\n             for x0 in (xs[0], xs[1])][:4]\\n    views.append(x[:, :, ys[2]:ys[2] + crop, xs[2]:xs[2] + crop])\\n    return views[:5]\\n\\n\\ndef views_multiscale(x, sizes):\\n    \\"\\"\\"Resize batch về từng kích thước trong sizes (CNN có global pooling thì OK;\\n    ViT/Swin cần xử lý pos-embed/cửa sổ -> có thể lỗi, ghi rõ giới hạn).\\"\\"\\"\\n    return [F.interpolate(x, size=(s, s), mode=\\"bilinear\\", align_corners=False) for s in sizes]\\n\\n\\ndef _softmax(z):\\n    z = z - z.max(1, keepdims=True)\\n    e = np.exp(z)\\n    return e / e.sum(1, keepdims=True)\\n\\n\\ndef aggregate_views(logits_per_view, space: str = \\"prob\\"):\\n    \\"\\"\\"Gộp K view: prob = mean softmax; logit = softmax(mean logit). Trả về xác suất.\\"\\"\\"\\n    assert space in (\\"prob\\", \\"logit\\"), space\\n    arr = [np.asarray(l, dtype=np.float64) for l in logits_per_view]\\n    if space == \\"prob\\":\\n        return np.mean([_softmax(l) for l in arr], axis=0)\\n    return _softmax(np.mean(arr, axis=0))\\n\\n\\ndef ensemble_probs(list_of_probs):\\n    \\"\\"\\"Trung bình xác suất nhiều mô hình (cùng tập ảnh, cùng thứ tự file).\\"\\"\\"\\n    arr = [np.asarray(p, dtype=np.float64) for p in list_of_probs]\\n    out = np.mean(arr, axis=0)\\n    return out / out.sum(1, keepdims=True)\\n\\n\\ndef fit_temperature(val_logits, val_labels) -> float:\\n    \\"\\"\\"Tìm T>0 cực tiểu NLL trên VAL (LBFGS trên logT). Accuracy không đổi. KHÔNG khớp trên test.\\"\\"\\"\\n    z = torch.tensor(np.asarray(val_logits), dtype=torch.float32)\\n    y = torch.tensor(np.asarray(val_labels), dtype=torch.long)\\n    logT = torch.zeros((), requires_grad=True)\\n    opt = torch.optim.LBFGS([logT], lr=0.5, max_iter=100)\\n\\n    def closure():\\n        opt.zero_grad()\\n        loss = F.cross_entropy(z / logT.exp(), y)\\n        loss.backward()\\n        return loss\\n    opt.step(closure)\\n    T = float(logT.exp().item())\\n    return min(max(T, 0.05), 10.0)\\n\\n\\ndef apply_temperature(logits, T: float):\\n    return _softmax(np.asarray(logits, dtype=np.float64) / float(T))\\n\\n\\ndef fuse_conv_bn(model):\\n    \\"\\"\\"Gộp từng cặp Conv2d+BatchNorm2d liền kề (model.eval trước). Trả về model.\\n    ViT/Swin/ConvNeXt dùng LayerNorm -> không áp dụng, ghi rõ.\\"\\"\\"\\n    model.eval()\\n    fused = 0\\n    max_err = 0.0\\n    for parent in list(model.modules()):\\n        children = list(parent.named_children())\\n        for (n1, m1), (n2, m2) in zip(children, children[1:]):\\n            if isinstance(m1, nn.Conv2d) and isinstance(m2, nn.BatchNorm2d):\\n                with torch.no_grad():\\n                    w = m1.weight\\n                    gamma = m2.weight if m2.weight is not None else torch.ones(m2.num_features)\\n                    beta = m2.bias if m2.bias is not None else torch.zeros(m2.num_features)\\n                    mean, var, eps = m2.running_mean, m2.running_var, m2.eps\\n                    s = gamma / torch.sqrt(var + eps)\\n                    w_new = w * s.view(-1, 1, 1, 1)\\n                    b_new = beta + ( (m1.bias if m1.bias is not None else torch.zeros_like(mean)) - mean) * s\\n                    conv = nn.Conv2d(m1.in_channels, m1.out_channels, m1.kernel_size, m1.stride,\\n                                     m1.padding, m1.dilation, m1.groups, True, m1.padding_mode).to(w.device)\\n                    conv.weight.copy_(w_new)\\n                    conv.bias.copy_(b_new)\\n                    setattr(parent, n1, conv)\\n                    setattr(parent, n2, nn.Identity())\\n                    fused += 1\\n    if fused == 0:\\n        print(\\"fuse_conv_bn: không tìm thấy cặp Conv+BN (có thể là ViT/Swin/LayerNorm) -> giữ nguyên\\")\\n    else:\\n        print(f\\"fuse_conv_bn: đã gộp {fused} cặp Conv+BN\\")\\n    return model\\n", "code/losses.py": "\\"\\"\\"losses.py - loss và trộn mẫu (Mixup/CutMix).\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\n\\nclass LabelSmoothingCE(nn.Module):\\n    \\"\\"\\"CE với label smoothing: q\'(k)=(1-eps)*1[k==y]+eps/K. eps=0 -> đúng CE.\\"\\"\\"\\n\\n    def __init__(self, smoothing: float = 0.1):\\n        super().__init__()\\n        self.smoothing = float(smoothing)\\n\\n    def forward(self, logits, target):\\n        return F.cross_entropy(logits, target, label_smoothing=self.smoothing)\\n\\n\\nclass FocalLoss(nn.Module):\\n    \\"\\"\\"Focal loss: FL(p_t) = -alpha_t*(1-p_t)^gamma*log(p_t). gamma=0 -> đúng CE (sai số <1e-6).\\"\\"\\"\\n\\n    def __init__(self, gamma: float = 2.0, alpha=None):\\n        super().__init__()\\n        self.gamma = float(gamma)\\n        if alpha is not None:\\n            alpha = torch.as_tensor(alpha, dtype=torch.float32)\\n        self.register_buffer(\\"alpha\\", alpha)\\n\\n    def forward(self, logits, target):\\n        logp = F.log_softmax(logits, dim=1)\\n        logp_t = logp.gather(1, target.view(-1, 1)).squeeze(1)\\n        p_t = logp_t.exp()\\n        mod = (1.0 - p_t) ** self.gamma\\n        if self.alpha is not None:\\n            at = self.alpha.to(logits.device)[target]\\n            loss = -at * mod * logp_t\\n        else:\\n            loss = -mod * logp_t\\n        return loss.mean()\\n\\n\\ndef build_criterion(kind: str = \\"ce\\", **kw):\\n    \\"\\"\\"kind: ce | ls | focal | ce_weighted.\\"\\"\\"\\n    if kind == \\"ce\\":\\n        return nn.CrossEntropyLoss()\\n    if kind == \\"ls\\":\\n        return LabelSmoothingCE(smoothing=float(kw.get(\\"smoothing\\", 0.1)))\\n    if kind == \\"focal\\":\\n        return FocalLoss(gamma=float(kw.get(\\"gamma\\", 2.0)), alpha=kw.get(\\"alpha\\", None))\\n    if kind == \\"ce_weighted\\":\\n        w = kw.get(\\"weight\\", None)\\n        if w is not None:\\n            w = torch.as_tensor(w, dtype=torch.float32)\\n        return nn.CrossEntropyLoss(weight=w)\\n    raise ValueError(f\\"loss không hỗ trợ: {kind}\\")\\n\\n\\ndef class_weights(counts, beta: float = 0.0):\\n    \\"\\"\\"Trọng số lớp từ số ảnh TRAIN. beta=0: 1/n_c (chuẩn hoá mean=1); beta>0: class-balanced\\n    w=(1-beta)/(1-beta^n), chuẩn hoá tổng về số lớp. Chỉ dùng số liệu train.\\"\\"\\"\\n    import numpy as np\\n    n = np.asarray(list(counts), dtype=np.float64)\\n    assert (n > 0).all(), \\"mọi lớp phải có ảnh trong train\\"\\n    if beta and beta > 0:\\n        w = (1.0 - beta) / (1.0 - np.power(beta, n))\\n        w = w / w.sum() * len(n)\\n    else:\\n        w = (1.0 / n)\\n        w = w / w.mean()\\n    return torch.tensor(w, dtype=torch.float32)\\n\\n\\ndef mix_batch(x, y, alpha: float = 1.0, mode: str = \\"cutmix\\"):\\n    \\"\\"\\"Trộn batch. Trả về (x_mix, (y_a, y_b, lam)). lam~Beta(alpha,alpha).\\"\\"\\"\\n    assert mode in (\\"mixup\\", \\"cutmix\\"), mode\\n    lam = float(torch.distributions.Beta(alpha, alpha).sample(()))\\n    perm = torch.randperm(x.size(0), device=x.device)\\n    y_a, y_b = y, y[perm]\\n    if mode == \\"mixup\\":\\n        return lam * x + (1.0 - lam) * x[perm], (y_a, y_b, lam)\\n    # CutMix: hộp ngẫu nhiên, lam hiệu chỉnh theo diện tích thực sau khi cắt biên.\\n    W, H = x.size(-1), x.size(-2)\\n    cut_ratio = (1.0 - lam) ** 0.5\\n    cut_w, cut_h = int(W * cut_ratio), int(H * cut_ratio)\\n    cx, cy = torch.randint(W, (1,)).item(), torch.randint(H, (1,)).item()\\n    x1 = max(cx - cut_w // 2, 0); y1 = max(cy - cut_h // 2, 0)\\n    x2 = min(cx + cut_w // 2, W); y2 = min(cy + cut_h // 2, H)\\n    xm = x.clone()\\n    xm[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]\\n    lam = 1.0 - (x2 - x1) * (y2 - y1) / (W * H)\\n    return xm, (y_a, y_b, float(lam))\\n\\n\\ndef mixed_loss(criterion, logits, targets):\\n    \\"\\"\\"lam*crit(logits,y_a)+(1-lam)*crit(logits,y_b).\\"\\"\\"\\n    y_a, y_b, lam = targets\\n    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)\\n", "code/model.py": "\\"\\"\\"model.py - tạo backbone, đóng băng, nhóm tham số, đếm params/GMAC.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport torch\\nimport torch.nn as nn\\n\\nSUGGESTED_BACKBONES = {\\n    \\"resnet50\\": \\"resnet50\\",\\n    \\"resnext50\\": \\"resnext50_32x4d\\",\\n    \\"convnext_tiny\\": \\"convnext_tiny\\",\\n    \\"deit_small\\": \\"deit_small_patch16_224\\",\\n    \\"swin_tiny\\": \\"swin_tiny_patch4_window7_224\\",\\n    \\"efficientnet_b0\\": \\"efficientnet_b0\\",\\n    \\"mobilenetv3\\": \\"mobilenetv3_large_100\\",\\n}\\n\\n\\ndef _head_param_ids(model) -> set[int]:\\n    \\"\\"\\"Tập id() tham số của head/classifier mới (để tách LR head gấp 10).\\"\\"\\"\\n    head = None\\n    for attr in (\\"get_classifier\\",):\\n        if hasattr(model, attr):\\n            try:\\n                head = model.get_classifier()\\n                break\\n            except Exception:\\n                pass\\n    if head is None:\\n        for name in (\\"head\\", \\"fc\\", \\"classifier\\", \\"heads\\"):\\n            if hasattr(model, name):\\n                head = getattr(model, name)\\n                break\\n    ids: set[int] = set()\\n    if head is not None:\\n        for p in head.parameters():\\n            ids.add(id(p))\\n    return ids\\n\\n\\ndef build_model(name: str, pretrained: bool = True, num_classes: int = 9,\\n                drop_rate: float = 0.0, init: str = \\"finetune\\"):\\n    \\"\\"\\"Tạo model 9 lớp. init: scratch | frozen | finetune. Ghi tag trọng số vào model.pretrained_tag.\\"\\"\\"\\n    if init == \\"scratch\\":\\n        pretrained = False\\n    try:\\n        import timm\\n        model = timm.create_model(name, pretrained=pretrained, num_classes=num_classes,\\n                                  drop_rate=drop_rate)\\n        tag = getattr(getattr(model, \\"pretrained_cfg\\", None), \\"tag\\", None) or getattr(\\n            getattr(model, \\"pretrained_cfg\\", None), \\"url\\", \\"\\") or name\\n        model.pretrained_tag = str(tag)\\n    except ImportError:\\n        import torchvision.models as M\\n        if not hasattr(M, name):\\n            raise ValueError(f\\"không có backbone {name} (cần timm hoặc torchvision)\\")\\n        w = \\"DEFAULT\\" if pretrained else None\\n        model = M.__dict__[name](weights=w)\\n        in_f = model.fc.in_features if hasattr(model, \\"fc\\") else model.classifier[-1].in_features\\n        model.fc = nn.Linear(in_f, num_classes) if hasattr(model, \\"fc\\") else model.classifier\\n        if hasattr(model, \\"classifier\\") and not hasattr(model, \\"fc\\"):\\n            model.classifier[-1] = nn.Linear(in_f, num_classes)\\n        if drop_rate:\\n            model.dropout_rate = drop_rate\\n        model.pretrained_tag = f\\"torchvision:{name}:{w}\\"\\n    if init == \\"frozen\\":\\n        freeze_backbone(model)\\n    elif init not in (\\"finetune\\", \\"scratch\\", \\"frozen\\"):\\n        raise ValueError(f\\"init không hỗ trợ: {init}\\")\\n    return model\\n\\n\\ndef freeze_backbone(model) -> None:\\n    \\"\\"\\"Đóng băng mọi tham số trừ head. Lưu ý: phần đóng băng phải chạy ở eval (xem train loop).\\"\\"\\"\\n    head_ids = _head_param_ids(model)\\n    for p in model.parameters():\\n        p.requires_grad = id(p) in head_ids\\n    model._frozen_bn = True\\n\\n\\ndef param_groups(model, lr_backbone: float, lr_head: float, weight_decay: float):\\n    \\"\\"\\"3 nhóm (slide tr.52): backbone decay / backbone norm+bias (wd=0) / head.\\"\\"\\"\\n    head_ids = _head_param_ids(model)\\n    g_bb, g_nobias, g_head = [], [], []\\n    for name, p in model.named_parameters():\\n        if not p.requires_grad:\\n            continue\\n        if id(p) in head_ids:\\n            g_head.append(p)\\n        elif p.ndim <= 1:\\n            g_nobias.append(p)\\n        else:\\n            g_bb.append(p)\\n    groups = []\\n    if g_bb:\\n        groups.append({\\"params\\": g_bb, \\"lr\\": lr_backbone, \\"weight_decay\\": weight_decay})\\n    if g_nobias:\\n        groups.append({\\"params\\": g_nobias, \\"lr\\": lr_backbone, \\"weight_decay\\": 0.0})\\n    if g_head:\\n        groups.append({\\"params\\": g_head, \\"lr\\": lr_head, \\"weight_decay\\": weight_decay})\\n    assert groups, \\"không còn tham số train được (kiểm tra freeze_backbone)\\"\\n    return groups\\n\\n\\ndef count_params(model) -> float:\\n    \\"\\"\\"Số tham số (triệu), kể cả tham số đóng băng.\\"\\"\\"\\n    return sum(p.numel() for p in model.parameters()) / 1e6\\n\\n\\ndef count_gmacs(model, img_size: int = 224) -> float:\\n    \\"\\"\\"GMAC cho 1 ảnh 3xHxW. Thử thop -> fvcore -> ptflops, cuối cùng hook Conv2d/Linear.\\"\\"\\"\\n    device = next(model.parameters()).device\\n    x = torch.randn(1, 3, img_size, img_size, device=device)\\n    for lib in (\\"thop\\", \\"fvcore\\", \\"ptflops\\"):\\n        try:\\n            if lib == \\"thop\\":\\n                from thop import profile\\n                macs, _ = profile(model, inputs=(x,), verbose=False)\\n                return macs / 1e9\\n            elif lib == \\"fvcore\\":\\n                from fvcore.nn import FlopCountAnalysis\\n                return FlopCountAnalysis(model, x).total() / 2 / 1e9\\n            else:\\n                from ptflops import get_model_complexity_info\\n                model.eval()\\n                macs, _ = get_model_complexity_info(model, (3, img_size, img_size),\\n                                                    as_strings=False, verbose=False)\\n                return macs / 1e9\\n        except Exception:\\n            continue\\n    # Fallback: hook đếm Conv2d + Linear (MAC, không phải FLOPs 2x).\\n    macs = [0]\\n\\n    def conv_hook(m, inp, out):\\n        b, oc, oh, ow = out.shape\\n        ic = m.in_channels // m.groups\\n        kh, kw = m.kernel_size\\n        macs[0] += b * oc * oh * ow * ic * kh * kw\\n\\n    def lin_hook(m, inp, out):\\n        macs[0] += out.numel() * m.in_features  # batch*out*in\\n    hooks = []\\n    for m in model.modules():\\n        if isinstance(m, nn.Conv2d):\\n            hooks.append(m.register_forward_hook(conv_hook))\\n        elif isinstance(m, nn.Linear):\\n            hooks.append(m.register_forward_hook(lin_hook))\\n    was_train = model.training\\n    model.eval()\\n    with torch.inference_mode():\\n        model(x)\\n    for h in hooks:\\n        h.remove()\\n    if was_train:\\n        model.train()\\n    return macs[0] / 1e9\\n", "code/train.py": "\\"\\"\\"train.py - một hàm run(cfg) cho mọi thí nghiệm (B, T, F).\\n\\nChạy: python train.py --set exp_id=B01 backbone=resnet50 seed=0\\nCheckpoint chọn theo macro-F1 val (hòa lấy epoch sớm hơn). Test chỉ ghi khi save_test_predictions=True (Bước 4).\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport copy\\nimport dataclasses\\nimport json\\nimport math\\nimport random\\nimport sys\\nimport time\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport torch\\nimport torch.nn as nn\\nimport numpy as np  # noqa: E402  (torch phai import truoc numpy/pandas: tranh xung dot OpenMP/MKL tren Windows)\\nimport pandas as pd  # noqa: E402\\n\\n\\n@dataclass\\nclass Config:\\n    exp_id: str = \\"T00\\"\\n    seed: int = 0\\n    fold: int = 0\\n    backbone: str = \\"resnet50\\"\\n    init: str = \\"finetune\\"            # scratch | frozen | finetune\\n    drop_rate: float = 0.0\\n    img_size: int = 224\\n    aug: str = \\"basic\\"                # basic | color | trivial | randaug\\n    sampler: str | None = None        # None | balanced\\n    mix: str | None = None            # None | mixup | cutmix\\n    mix_alpha: float = 1.0\\n    loss: str = \\"ce\\"                  # ce | ls | focal | ce_weighted\\n    label_smoothing: float = 0.0\\n    focal_gamma: float = 2.0\\n    class_weight_beta: float | None = None\\n    epochs: int = 12\\n    batch_size: int = 64\\n    lr_backbone: float = 1e-4\\n    lr_head: float = 1e-3\\n    weight_decay: float = 0.05\\n    warmup_epochs: float = 1.0\\n    ema_decay: float | None = None\\n    amp: bool = True\\n    num_workers: int = 2\\n    images_dir: str = \\"data/images\\"\\n    labels_dir: str = \\"data/labels\\"\\n    out_dir: str = \\"runs\\"\\n    pred_dir: str = \\"predictions\\"\\n    save_test_predictions: bool = False\\n\\n\\ndef run_dir(cfg: Config) -> Path:\\n    return Path(cfg.out_dir) / cfg.exp_id / f\\"seed{cfg.seed}\\"\\n\\n\\ndef pred_path(cfg: Config, split: str) -> Path:\\n    return Path(cfg.pred_dir) / f\\"{cfg.exp_id}_seed{cfg.seed}_{split}.csv\\"\\n\\n\\ndef set_seed(seed: int) -> None:\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    torch.cuda.manual_seed_all(seed)\\n\\n\\ndef build_optimizer(model, cfg: Config):\\n    from model import param_groups\\n    groups = param_groups(model, cfg.lr_backbone, cfg.lr_head, cfg.weight_decay)\\n    return torch.optim.AdamW(groups)\\n\\n\\ndef build_scheduler(optimizer, cfg: Config, steps_per_epoch: int):\\n    \\"\\"\\"Warmup tuyến tính (warmup_epochs) rồi cosine về ~0, cập nhật theo bước.\\"\\"\\"\\n    total = max(cfg.epochs * steps_per_epoch, 1)\\n    warmup = int(round(cfg.warmup_epochs * steps_per_epoch))\\n\\n    def lr_fn(step):\\n        if step < max(warmup, 1):\\n            return (step + 1) / max(warmup, 1)\\n        t = (step - warmup) / max(total - warmup, 1)\\n        return 0.5 * (1.0 + math.cos(math.pi * min(max(t, 0.0), 1.0)))\\n    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)\\n\\n\\nclass EMA:\\n    \\"\\"\\"W_ema <- d*W_ema + (1-d)*W sau mỗi bước. Đánh giá bằng bản sao ema_model.\\"\\"\\"\\n\\n    def __init__(self, model, decay: float):\\n        self.decay = float(decay)\\n        self.ema_model = copy.deepcopy(model).eval()\\n        for p in self.ema_model.parameters():\\n            p.requires_grad_(False)\\n\\n    @torch.no_grad()\\n    def update(self, model) -> None:\\n        d = self.decay\\n        for e, m in zip(self.ema_model.parameters(), model.parameters()):\\n            e.mul_(d).add_(m.detach(), alpha=1.0 - d)\\n        for be, bm in zip(self.ema_model.buffers(), model.buffers()):\\n            be.copy_(bm)\\n\\n    def copy_to(self, model) -> None:\\n        model.load_state_dict(self.ema_model.state_dict())\\n\\n\\ndef train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, cfg: Config,\\n                    device, ema: EMA | None = None) -> dict:\\n    from losses import mix_batch, mixed_loss\\n    frozen = getattr(model, \\"_frozen_bn\\", False)\\n    model.train()\\n    if frozen:  # backbone đóng băng: giữ BN ở eval\\n        for m in model.modules():\\n            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):\\n                m.eval()\\n    tot, n, t0 = 0.0, 0, time.time()\\n    use_amp = cfg.amp and device.type == \\"cuda\\"\\n    for x, y, _ in loader:\\n        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\\n        optimizer.zero_grad(set_to_none=True)\\n        with torch.autocast(device.type, enabled=use_amp):\\n            if cfg.mix:\\n                xm, targets = mix_batch(x, y, cfg.mix_alpha, cfg.mix)\\n                ta, tb, lam = targets[0], targets[1], targets[2]\\n                loss = mixed_loss(criterion, model(xm),\\n                                  (ta, tb, lam if isinstance(lam, float) else float(lam)))\\n            else:\\n                loss = criterion(model(x), y)\\n        if scaler is not None:\\n            scaler.scale(loss).backward()\\n            scaler.step(optimizer)\\n            scaler.update()\\n        else:\\n            loss.backward()\\n            optimizer.step()\\n        scheduler.step()\\n        if ema is not None:\\n            ema.update(model)\\n        tot += loss.item() * x.size(0)\\n        n += x.size(0)\\n    return {\\"train_loss\\": tot / max(n, 1), \\"lr\\": optimizer.param_groups[0][\\"lr\\"],\\n            \\"time_s\\": time.time() - t0}\\n\\n\\n@torch.inference_mode()\\ndef evaluate(model, loader, criterion, device):\\n    model.eval()\\n    names, ys, outs, tot, n = [], [], [], 0.0, 0\\n    for x, y, fn in loader:\\n        x = x.to(device, non_blocking=True)\\n        logits = model(x)\\n        outs.append(logits.detach().cpu())\\n        ys.append(torch.as_tensor(y))\\n        names.extend(list(fn))\\n        if criterion is not None:\\n            tot += criterion(logits, y.to(logits.device)).item() * x.size(0)\\n            n += x.size(0)\\n    logits = torch.cat(outs).numpy()\\n    y_true = torch.cat(ys).numpy().astype(np.int64)\\n    return names, y_true, logits, (tot / max(n, 1))\\n\\n\\ndef plot_curves(history: list[dict], path: str | Path, title: str) -> None:\\n    import matplotlib.pyplot as plt\\n    h = pd.DataFrame(history)\\n    fig, ax = plt.subplots(1, 2, figsize=(11, 4))\\n    ax[0].plot(h[\\"epoch\\"], h[\\"train_loss\\"], label=\\"train loss\\")\\n    if \\"val_loss\\" in h:\\n        ax[0].plot(h[\\"epoch\\"], h[\\"val_loss\\"], label=\\"val loss\\")\\n    ax[0].set_xlabel(\\"epoch\\"); ax[0].set_ylabel(\\"loss\\"); ax[0].legend(); ax[0].grid(True, alpha=0.3)\\n    ax[1].plot(h[\\"epoch\\"], h[\\"val_macro_f1\\"], marker=\\"o\\", label=\\"val macro-F1\\")\\n    ax[1].set_xlabel(\\"epoch\\"); ax[1].set_ylabel(\\"macro-F1\\"); ax[1].legend(); ax[1].grid(True, alpha=0.3)\\n    fig.suptitle(title)\\n    fig.tight_layout()\\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\\n    fig.savefig(path, dpi=150)\\n    plt.close(fig)\\n\\n\\ndef _repo_root() -> Path:\\n    # code/ nằm trong submissions/<mssv>/code -> repo root cách 3 cấp.\\n    here = Path(__file__).resolve()\\n    for p in [here.parent, *here.parents]:\\n        if (p / \\"eval.py\\").exists():\\n            return p\\n    return Path.cwd()\\n\\n\\ndef run(cfg: Config) -> dict:\\n    import dataset as D\\n    import model as M\\n    import losses as L\\n    sys.path.insert(0, str(_repo_root()))\\n    from eval import compute_metrics, save_predictions\\n\\n    set_seed(cfg.seed)\\n    rdir = run_dir(cfg)\\n    rdir.mkdir(parents=True, exist_ok=True)\\n    (rdir / \\"config.json\\").write_text(json.dumps(dataclasses.asdict(cfg), indent=2, default=str))\\n\\n    train_df, val_df, test_df = D.load_split(cfg.labels_dir, cfg.fold)\\n    D.check_split(train_df, val_df, test_df, cfg.images_dir)\\n\\n    train_loader = D.make_loader(train_df, cfg.images_dir, D.build_transforms(True, cfg.img_size, cfg.aug),\\n                                 cfg.batch_size, True, cfg.sampler, cfg.num_workers)\\n    val_loader = D.make_loader(val_df, cfg.images_dir, D.build_transforms(False, cfg.img_size),\\n                               cfg.batch_size, False, None, cfg.num_workers)\\n    test_loader = (D.make_loader(test_df, cfg.images_dir, D.build_transforms(False, cfg.img_size),\\n                                 cfg.batch_size, False, None, cfg.num_workers)\\n                   if cfg.save_test_predictions else None)\\n\\n    device = torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n    torch.backends.cudnn.benchmark = True\\n    model = M.build_model(cfg.backbone, True, 9, cfg.drop_rate, cfg.init).to(device)\\n    if device.type == \\"cuda\\":\\n        try:\\n            model = model.to(memory_format=torch.channels_last)\\n        except Exception:\\n            pass\\n\\n    if cfg.loss == \\"ls\\":\\n        criterion = L.build_criterion(\\"ls\\", smoothing=cfg.label_smoothing or 0.1)\\n    elif cfg.loss == \\"focal\\":\\n        criterion = L.build_criterion(\\"focal\\", gamma=cfg.focal_gamma)\\n    elif cfg.loss == \\"ce_weighted\\":\\n        counts = train_df[\\"Label\\"].value_counts().sort_index().reindex(range(9), fill_value=1)\\n        w = L.class_weights(counts.tolist(), cfg.class_weight_beta or 0.0).to(device)\\n        criterion = L.build_criterion(\\"ce_weighted\\", weight=w)\\n    else:\\n        criterion = L.build_criterion(\\"ce\\")\\n\\n    opt = build_optimizer(model, cfg)\\n    sch = build_scheduler(opt, cfg, max(len(train_loader), 1))\\n    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device.type == \\"cuda\\"))\\n    ema = EMA(model, cfg.ema_decay) if cfg.ema_decay else None\\n    eval_model = ema.ema_model if ema else model\\n\\n    history: list = []\\n    best_f1, best_ep, best_state = -1.0, -1, None\\n    t_epoch = []\\n    for epoch in range(cfg.epochs):\\n        tr = train_one_epoch(model, train_loader, criterion, opt, sch, scaler, cfg, device, ema)\\n        fn, yt, logits, vl = evaluate(eval_model, val_loader, criterion, device)\\n        probs = torch.softmax(torch.from_numpy(logits), 1).numpy()\\n        m = compute_metrics(yt, probs.argmax(1), probs)\\n        t_epoch.append(tr[\\"time_s\\"])\\n        history.append({\\"epoch\\": epoch, \\"train_loss\\": tr[\\"train_loss\\"], \\"val_loss\\": vl,\\n                        \\"val_macro_f1\\": m[\\"macro_f1\\"], \\"val_top1\\": m[\\"top1\\"], \\"lr\\": tr[\\"lr\\"]})\\n        print(f\\"{cfg.exp_id} seed={cfg.seed} epoch={epoch+1}/{cfg.epochs} \\"\\n              f\\"loss={tr[\'train_loss\']:.4f} val_f1={m[\'macro_f1\']:.4f} \\"\\n              f\\"train_s={tr[\'time_s\']:.1f}\\", flush=True)\\n        pd.DataFrame(history).to_csv(rdir / \\"history.csv\\", index=False)\\n        if m[\\"macro_f1\\"] > best_f1 + 1e-12:  # hòa -> giữ epoch sớm hơn\\n            best_f1, best_ep = m[\\"macro_f1\\"], epoch\\n            best_state = copy.deepcopy(eval_model.state_dict())\\n            torch.save({\\"epoch\\": epoch, \\"state\\": best_state, \\"cfg\\": dataclasses.asdict(cfg)},\\n                       rdir / \\"best.pt\\")\\n    eval_model.load_state_dict(best_state)\\n    torch.save({\\"epoch\\": best_ep, \\"state\\": best_state, \\"cfg\\": dataclasses.asdict(cfg)}, rdir / \\"best.pt\\")\\n\\n    fn, yt, logits, _ = evaluate(eval_model, val_loader, None, device)\\n    probs = torch.softmax(torch.from_numpy(logits), 1).numpy()\\n    save_predictions(pred_path(cfg, \\"val\\"), fn, yt, probs)\\n    vm = compute_metrics(yt, probs.argmax(1), probs)\\n    summary = {\\"best_epoch\\": best_ep, \\"val_macro_f1\\": vm[\\"macro_f1\\"], \\"val_top1\\": vm[\\"top1\\"],\\n               \\"params_M\\": M.count_params(eval_model), \\"train_s_per_epoch\\": float(np.mean(t_epoch)),\\n               \\"tag\\": getattr(eval_model, \\"pretrained_tag\\", cfg.backbone)}\\n    try:\\n        summary[\\"gmacs\\"] = M.count_gmacs(eval_model, cfg.img_size)\\n    except Exception as e:\\n        summary[\\"gmacs\\"] = None\\n        summary[\\"gmacs_note\\"] = str(e)\\n    if test_loader is not None:  # chỉ Bước 4, đúng một lần mỗi seed\\n        fn, yt, logits, _ = evaluate(eval_model, test_loader, None, device)\\n        probs = torch.softmax(torch.from_numpy(logits), 1).numpy()\\n        save_predictions(pred_path(cfg, \\"test\\"), fn, yt, probs)\\n    (rdir / \\"summary.json\\").write_text(json.dumps(summary, indent=2, default=str))\\n    plot_curves(history, Path(\\"curves\\") / f\\"{cfg.exp_id}_{cfg.backbone}_s{cfg.seed}.png\\",\\n                f\\"{cfg.exp_id} {cfg.backbone} seed{cfg.seed}\\")\\n    print(json.dumps(summary, indent=2))\\n    return summary\\n\\n\\ndef parse_overrides(pairs: list[str]) -> dict:\\n    fields = {f.name: f.type for f in dataclasses.fields(Config)}\\n    out = {}\\n    for p in pairs:\\n        if \\"=\\" not in p:\\n            raise ValueError(f\\"override phải dạng KEY=VALUE, nhận {p!r}\\")\\n        k, v = p.split(\\"=\\", 1)\\n        if k not in fields:\\n            raise ValueError(f\\"Config không có trường {k!r}\\")\\n        t = str(fields[k])\\n        if v in (\\"none\\", \\"None\\", \\"null\\") and \\"None\\" in t:\\n            out[k] = None\\n        elif v in (\\"true\\", \\"True\\"):\\n            out[k] = True\\n        elif v in (\\"false\\", \\"False\\"):\\n            out[k] = False\\n        elif \\"int\\" in t and \\"float\\" not in t:\\n            out[k] = int(v)\\n        elif \\"float\\" in t:\\n            out[k] = float(v)\\n        else:\\n            try:\\n                out[k] = int(v)\\n            except ValueError:\\n                try:\\n                    out[k] = float(v)\\n                except ValueError:\\n                    out[k] = v\\n        if k in (\\"epochs\\", \\"batch_size\\", \\"img_size\\", \\"num_workers\\") and isinstance(out[k], float):\\n            out[k] = int(out[k])\\n    return out\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"--set\\", nargs=\\"*\\", default=[], help=\\"override KEY=VALUE\\")\\n    args = ap.parse_args()\\n    cfg = Config(**parse_overrides(args.set))\\n    print(run(cfg))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for name, source in PAYLOAD.items():
    dest = Path(name)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(source, encoding='utf-8')
for folder in ('curves', 'predictions', 'runs', 'eval_out'):
    Path(folder).mkdir(exist_ok=True)
print('Embedded code and eval.py ready')


In [ ]:
%pip -q install timm openpyxl
import sys, os, platform
sys.path.insert(0, "code")
sys.path.insert(0, ".")
import torch
try:
    import timm; print("timm", timm.__version__)
except ImportError:
    print("THIEU timm")
print("python", platform.python_version(), "| torch", torch.__version__)
assert torch.cuda.is_available(), "GPU runtime required"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO GPU -> doi Runtime sang GPU!")
try:
    import eval as ev; print("eval.py OK")
except ImportError:
    print("THIEU eval.py: upload file eval.py (goc repo) vao cung thu muc voi code/ roi chay lai")

## 0b. Lay data.zip tu Drive (tuy chon — bo qua neu de muc 1 tu tai)

Up `data.zip` len Drive truoc (vd `MyDrive/data.zip`), chay cell duoi de mount + copy ve. Neu khong, bo qua cell nay.

In [ ]:
print('Automatic Zenodo download; no Drive mount required')


## 1. Tai du lieu + tim IMAGES_DIR tu dong (tu bo qua neu da upload du)

In [ ]:
import hashlib, glob, urllib.request, shutil
os.makedirs("data/labels", exist_ok=True)
jpgs_exist = glob.glob("data/**/*.jpg", recursive=True)
if len(jpgs_exist) >= 17000:
    print("Da co san", len(jpgs_exist), "anh (do upload) -> bo qua tai Zenodo")
else:
    print("Chua du anh (%d) -> tai tu Zenodo..." % len(jpgs_exist))
    if not os.path.exists("data/images.zip"):
        urllib.request.urlretrieve("https://zenodo.org/records/7939060/files/images.zip?download=1", "data/images.zip")
    h = hashlib.md5()
    with open("data/images.zip", "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", "MD5 sai: " + h.hexdigest()
    print("MD5 OK")
    shutil.unpack_archive("data/images.zip", "data/")
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    dst = "data/labels/" + name + ".csv"
    if not os.path.exists(dst):
        urllib.request.urlretrieve(BASE + "/" + name + ".csv", dst)
print(sorted(os.listdir("data/labels")))
from collections import Counter
jpgs = glob.glob("data/**/*.jpg", recursive=True)
assert jpgs, "khong thay jpg trong data/"
IMAGES_DIR = Counter(os.path.dirname(j) for j in jpgs).most_common(1)[0][0]
LABELS_DIR = "data/labels"
print("IMAGES_DIR =", IMAGES_DIR, "| so jpg =", len(jpgs))

## 2. Buoc 0 — EDA + kiem tra split (bat buoc)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
import dataset as D
tr_df, va_df, te_df = D.load_split(LABELS_DIR, 0)
info = D.check_split(tr_df, va_df, te_df, IMAGES_DIR)
print(info["n"], "total =", info["total"])
names = ["Chinee apple","Lantana","Parkinsonia","Parthenium","Prickly acacia","Rubber vine","Siam weed","Snake weed","Negative"]
tot = [info["per_class"]["train"].get(i,0)+info["per_class"]["val"].get(i,0)+info["per_class"]["test"].get(i,0) for i in range(9)]
paper = [1125,1064,1031,1022,1062,1009,1074,1016,9106]
for i,n in enumerate(names): print(n, "dem =", tot[i], "| bao =", paper[i])
pd.DataFrame({"train": [info["per_class"]["train"].get(i,0) for i in range(9)],
              "val": [info["per_class"]["val"].get(i,0) for i in range(9)],
              "test": [info["per_class"]["test"].get(i,0) for i in range(9)]}, index=names).plot.bar(figsize=(10,4))
plt.title("Phan bo lop train/val/test (fold 0)"); plt.tight_layout(); plt.savefig("curves/eda_perclass.png", dpi=120); plt.show()

### Xem anh mau (3 anh / lop)

In [ ]:
from PIL import Image
fig, ax = plt.subplots(9, 3, figsize=(6, 14))
for i in range(9):
    samp = tr_df[tr_df.Label == i].head(3).Filename.tolist()
    for j, fn in enumerate(samp):
        ax[i, j].imshow(Image.open(os.path.join(IMAGES_DIR, fn)).convert("RGB"))
        ax[i, j].axis("off")
        if j == 0: ax[i, j].set_ylabel(names[i], fontsize=8)
plt.tight_layout(); plt.savefig("curves/eda_samples.png", dpi=120); plt.show()

### Kiem tra pipeline: loss ban dau ~2.197, overfit 1 batch, anh sau augmentation

In [ ]:
import torch, numpy as np
from train import set_seed
import model as M, dataset as D
set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = M.build_model("resnet50", True, 9, 0.0, "finetune").to(device)
tf_val = D.build_transforms(False, 224)
loader = D.make_loader(va_df.head(64), IMAGES_DIR, tf_val, 16, False, None, 0)
x, y, _ = next(iter(loader)); x, y = x.to(device), y.to(device)
net.eval()
with torch.inference_mode(): l0 = torch.nn.CrossEntropyLoss()(net(x), y).item()
print("loss ban dau =", round(l0,3), "(ky vong ~2.197 = -ln(1/9))")
tiny = D.make_loader(tr_df.head(8), IMAGES_DIR, D.build_transforms(True, 224, "basic"), 8, True, None, 0)
opt = torch.optim.Adam([p for p in net.parameters() if p.requires_grad], lr=1e-2)
net.train()
for it in range(40):
    xb, yb, _ = next(iter(tiny)); xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(); loss = torch.nn.CrossEntropyLoss()(net(xb), yb); loss.backward(); opt.step()
print("loss sau overfit 8 anh =", round(loss.item(),4), "(ky vong gan 0)")
mean = np.array(D.IMAGENET_MEAN).reshape(3,1,1); std = np.array(D.IMAGENET_STD).reshape(3,1,1)
fig, ax = plt.subplots(1, 4, figsize=(10,3))
xb, yb, _ = next(iter(tiny))
for j in range(4):
    img = (xb[j].numpy()*std+mean).transpose(1,2,0).clip(0,1)
    ax[j].imshow(img); ax[j].set_title(names[int(yb[j])], fontsize=8); ax[j].axis("off")
plt.tight_layout(); plt.savefig("curves/eda_aug.png", dpi=120); plt.show()

## 3. Buoc 1 — So sanh backbone (>=5, cung cong thuc nen, cung seed)

In [ ]:
FULL = True  # doi True de chay that 12 epoch
EPOCHS = 12 if FULL else 2
from train import Config, run as original_run
import gc, json, dataclasses
def run(cfg):
    folder = Path(cfg.out_dir) / cfg.exp_id / f'seed{cfg.seed}'
    if (folder/'summary.json').exists() and (folder/'config.json').exists():
        previous = json.load(open(folder/'config.json'))
        if previous == dataclasses.asdict(cfg):
            print('Reuse completed:', cfg.exp_id, cfg.seed, flush=True)
            return json.load(open(folder/'summary.json'))
    gc.collect(); torch.cuda.empty_cache()
    return original_run(cfg)
del net, opt, x, y, xb, yb
gc.collect(); torch.cuda.empty_cache()
BACKBONES = ["resnet50","resnext50_32x4d","convnext_tiny","deit_small_patch16_224",
             "swin_tiny_patch4_window7_224","efficientnet_b0","mobilenetv3_large_100"]
rows = []
for i, bb in enumerate(BACKBONES):
    eid = "B%02d" % (i+1)
    try:
        s = run(Config(exp_id=eid, backbone=bb, seed=0, epochs=EPOCHS, batch_size=32,
                        images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
        s.update(exp_id=eid, backbone=bb); rows.append(s); print(eid, bb, "OK", round(s["val_macro_f1"],4))
    except Exception as e:
        print(eid, bb, "LOI:", type(e).__name__, str(e)[:200])
df1 = pd.DataFrame(rows); df1.to_csv("step1_summary.csv", index=False)
print(df1[["exp_id","backbone","params_M","gmacs","val_macro_f1","val_top1","train_s_per_epoch"]].to_string())

## 4. Buoc 2 — Cong thuc huan luyen (>=3 truc, moi lan khac nen 1 yeu to)

In [ ]:
BEST_BB = df1.sort_values("val_macro_f1", ascending=False).iloc[0]["backbone"]
print("Selected backbone on validation:", BEST_BB)
BASE_CFG = dict(backbone=BEST_BB, seed=0, epochs=EPOCHS, batch_size=32,
                images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
run(Config(exp_id="T00", **BASE_CFG))
EXPS = [("T01", dict(init="scratch")), ("T02", dict(init="frozen")),
        ("T03", dict(aug="color")), ("T04", dict(aug="trivial")),
        ("T05", dict(mix="cutmix", mix_alpha=1.0)),
        ("T06", dict(loss="ls", label_smoothing=0.1)),
        ("T07", dict(loss="focal", focal_gamma=2.0)),
        ("T08", dict(loss="ce_weighted")), ("T09", dict(sampler="balanced")),
        ("T10", dict(lr_backbone=1e-3)), ("T11", dict(ema_decay=0.999))]
for eid, kw in EXPS:
    try:
        run(Config(exp_id=eid, **dict(BASE_CFG, **kw)))
    except Exception as e:
        print(eid, "LOI:", type(e).__name__, str(e)[:200])
print("Xong. So sanh macro-F1 val cua T01-T11 voi T00 trong runs/*/summary.json")

## 5. Buoc 3 — Suy luan (>=4 PP) + do tre dung cach

In [ ]:
import json, torch, numpy as np
import dataset as D, model as M
from inference import predict_logits, view_hflip, views_multicrop, views_multiscale, aggregate_views, fit_temperature, apply_temperature, fuse_conv_bn
from benchmark import latency_report
from eval import compute_metrics
from torchvision import transforms as Tv
EXP = max(["T00"] + [eid for eid, _ in EXPS], key=lambda eid: json.load(open("runs/"+eid+"/seed0/summary.json"))["val_macro_f1"])
print("Selected training recipe on validation:", EXP)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tr_df, va_df, te_df = D.load_split(LABELS_DIR, 0)
valloader = D.make_loader(va_df, IMAGES_DIR, D.build_transforms(False, 224), 64, False, None, 2)
ckpt = torch.load("runs/" + EXP + "/seed0/best.pt", map_location=device)
cfg = ckpt["cfg"]
net = M.build_model(cfg["backbone"], False, 9, cfg.get("drop_rate", 0.0), "finetune")
net.load_state_dict(ckpt["state"]); net.to(device).eval()
names, yt, logits = predict_logits(net, valloader, device)
sm = lambda z: (lambda e: e/e.sum(1, keepdims=True))(np.exp(z-z.max(1, keepdims=True)))
base = compute_metrics(yt, logits.argmax(1), sm(logits))
print("I00 1-view: F1=%.4f top1=%.4f ECE=%.4f" % (base["macro_f1"], base["top1"], base["ece"]))
fn_f, _, logits_f = predict_logits(net, valloader, device, view_hflip)
assert fn_f == names
for space in ("prob", "logit"):
    p = aggregate_views([logits, logits_f], space)
    m = compute_metrics(yt, p.argmax(1), p)
    print("I01 flip-%s: F1=%.4f top1=%.4f (K=2, ~2x tre)" % (space, m["macro_f1"], m["top1"]))
tf256 = Tv.Compose([Tv.Resize(256), Tv.ToTensor(), Tv.Normalize(mean=list(D.IMAGENET_MEAN), std=list(D.IMAGENET_STD))])
loader256 = D.make_loader(va_df, IMAGES_DIR, tf256, 64, False, None, 2)
allp = []
with torch.inference_mode():
    for x, y, fn in loader256:
        x = x.to(device)
        ls = [net(v).detach().cpu().numpy() for v in views_multicrop(x, 224)]
        allp.append(aggregate_views(ls, "prob"))
p5 = np.concatenate(allp)
m5 = compute_metrics(yt, p5.argmax(1), p5)
print("I02 5-crop: F1=%.4f top1=%.4f (K=5, ~5x tre)" % (m5["macro_f1"], m5["top1"]))
try:
    allm = []
    with torch.inference_mode():
        for x, y, fn in loader256:
            x = x.to(device)
            ls = [net(v).detach().cpu().numpy() for v in views_multiscale(x, [224, 256, 288])]
            allm.append(aggregate_views(ls, "prob"))
    pm = np.concatenate(allm)
    mm = compute_metrics(yt, pm.argmax(1), pm)
    print("I04 multiscale 224/256/288: F1=%.4f top1=%.4f" % (mm["macro_f1"], mm["top1"]))
except Exception as e:
    print("I04 multiscale bo qua:", type(e).__name__, str(e)[:150], "(ViT/Swin can xu ly pos-embed)")
T = fit_temperature(logits, yt)
pcal = apply_temperature(logits, T)
print("I07 temperature: T=%.3f ECE truoc=%.4f sau=%.4f (acc khong doi)" % (T, base["ece"], compute_metrics(yt, pcal.argmax(1), pcal)["ece"]))
json.dump({"T": T, "EXP": EXP, "cfg": cfg}, open("temp_scaling.json", "w"))
import copy
net_bf = copy.deepcopy(net).eval()
net_fused = fuse_conv_bn(copy.deepcopy(net).eval())
xb, _, _ = next(iter(valloader)); xb = xb.to(device)
with torch.inference_mode():
    dmax = (net_bf(xb) - net_fused(xb)).abs().max().item()
print("I08 gop BN: sai so lon nhat truoc/sau = %.2e (ky vong <=1e-5; kien truc LayerNorm thi bo qua)" % dmax)
lat = {}
dev = "cuda" if torch.cuda.is_available() else "cpu"
for bs in (1, 32):
    for dt in (["fp32", "amp"] if torch.cuda.is_available() else ["fp32"]):
        r = latency_report(net, bs, 224, dt, dev)
        lat[str(bs)+"_"+dt] = r
        print("batch=%d %s: p50=%.1f p95=%.1f p99=%.1f ms (%.0f anh/s) [%s]" % (bs, dt, r["p50"], r["p95"], r["p99"], r["images_per_s"], r["gpu"]))
json.dump({"EXP": EXP, "I00": {"macro_f1": base["macro_f1"], "top1": base["top1"], "ece": base["ece"]},
           "latency": lat, "T": T}, open("inference_results.json", "w"))

## 6. Buoc 4 — Chung ket: >=3 seed, test 1 lan/seed (sau buoc nay KHONG sua cau hinh)

In [ ]:
import dataclasses, gc, json, torch, numpy as np
import dataset as D, model as M
from train import Config
from inference import predict_logits, fit_temperature, apply_temperature
from eval import save_predictions
selected = json.load(open('runs/'+EXP+'/seed0/config.json'))
FINAL_CFG = {k: v for k, v in selected.items() if k not in ('exp_id', 'seed', 'save_test_predictions')}
FINAL_CFG.update(epochs=12, save_test_predictions=False)
BASE_FINAL = dict(backbone=BEST_BB, epochs=12, batch_size=32, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, save_test_predictions=False)
json.dump({'selected_experiment': EXP, 'final': FINAL_CFG, 'baseline': BASE_FINAL}, open('final_selection.json', 'w'), indent=2)
device = torch.device('cuda')
tr_df, va_df, te_df = D.load_split(LABELS_DIR, 0)
valloader = D.make_loader(va_df, IMAGES_DIR, D.build_transforms(False, 224), 32, False, None, 2)
testloader = D.make_loader(te_df, IMAGES_DIR, D.build_transforms(False, 224), 32, False, None, 2)
calibration = {}
for exp, recipe in [('F01', FINAL_CFG), ('BASE12', BASE_FINAL)]:
    for seed in (0, 1, 2):
        run(Config(exp_id=exp, seed=seed, **recipe))
        ck = torch.load(f'runs/{exp}/seed{seed}/best.pt', map_location='cpu', weights_only=True)
        c = ck['cfg']
        n = M.build_model(c['backbone'], False, 9, c.get('drop_rate', 0.0), 'finetune')
        n.load_state_dict(ck['state']); del ck
        n.to(device).eval()
        fv, yv, lv = predict_logits(n, valloader, device)
        temperature = fit_temperature(lv, yv) if exp == 'F01' else 1.0
        calibration[f'{exp}_seed{seed}'] = temperature
        save_predictions(f'predictions/{exp}_seed{seed}_val.csv', fv, yv, apply_temperature(lv, temperature))
        # Exactly one test forward pass per model; derive both output versions from the same logits.
        ft, yt, lt = predict_logits(n, testloader, device)
        save_predictions(f'predictions/{exp}uncal_seed{seed}_test.csv', ft, yt, apply_temperature(lt, 1.0))
        save_predictions(f'predictions/{exp}_seed{seed}_test.csv', ft, yt, apply_temperature(lt, temperature))
        print(exp, 'seed', seed, 'test completed; temperature', temperature, flush=True)
        del n
        gc.collect(); torch.cuda.empty_cache()
json.dump(calibration, open('final_temperatures.json', 'w'), indent=2)


### Diem theo dung dinh nghia cua lop (eval.py)

In [ ]:
!python eval.py score --pred "predictions/F01_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out
!python eval.py score --pred "predictions/BASE12_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag BASE12 --out eval_out
!python eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/BASE12_seed*_test.csv" --uncal "predictions/F01uncal_seed*_test.csv" --final-val "predictions/F01_seed*_val.csv" --test-csv data/labels/test_subset0.csv --val-csv data/labels/val_subset0.csv --labels data/labels/labels.csv --out eval_out
# Neu co p95 batch-1 do dung cach: chay them lenh grade voi --latency-p95-ms <so> (lay tu Buoc 3).

## 7. Buoc 5 — Dong goi san pham + tai ve

In [ ]:
import json, glob, pandas as pd, zipfile
from pathlib import Path
rows = []
for s in sorted(glob.glob('runs/*/seed*/summary.json')):
    d = json.load(open(s)); d['path'] = s
    d.update(json.load(open(Path(s).parent / 'config.json')))
    rows.append(d)
pd.DataFrame(rows).to_csv('all_runs_summary.csv', index=False)
with pd.ExcelWriter('colab_results.xlsx') as writer:
    pd.DataFrame(rows).to_excel(writer, sheet_name='Runs', index=False)
    df1.to_excel(writer, sheet_name='Backbones', index=False)
    if Path('inference_results.json').exists():
        pd.DataFrame(json.load(open('inference_results.json'))['latency']).T.to_excel(writer, sheet_name='Latency')
with zipfile.ZipFile('colab_outputs.zip', 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for folder in ('code', 'predictions', 'curves', 'eval_out', 'runs'):
        for p in Path(folder).rglob('*'):
            if p.is_file() and p.suffix in ('.py', '.json', '.csv', '.png'):
                z.write(p, str(p))
    for pattern in ('*.csv', '*.json', '*.xlsx', 'eval.py'):
        for p in Path('.').glob(pattern): z.write(p, str(p))
print('DONE: colab_outputs.zip includes code, predictions, curves, evaluation, summaries and workbook', flush=True)
from google.colab import files
files.download('colab_outputs.zip')
